In [ ]:
!pip install ultralytics --no-cache-dir -q

In [ ]:
import os

dataset_dir = '/kaggle/input/datasets/anishpatil2853/helmate-detection'

print(f"Scanning directory: {dataset_dir}\n")
print("-" * 50)

# Walk through the directory and print folders and first 5 files in each
for root, dirs, files in os.walk(dataset_dir):
    level = root.replace(dataset_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}[Folder] {os.path.basename(root)}/')
    
    # Print first 5 files in this folder
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')
    
    # If there are more than 5 files, indicate it
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')
    print("-" * 50)

In [ ]:
import os
import glob

# Search the ENTIRE /kaggle/input/ directory for images or zip files
print("Searching for images (.png, .jpg) and .zip files in /kaggle/input/...\n")

# Find all image files
images = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG'):
    images.extend(glob.glob(os.path.join('/kaggle/input/', '**/' + ext), recursive=True))

# Find all zip files
zips = glob.glob(os.path.join('/kaggle/input/', '**/*.zip'), recursive=True)

print(f"Total Images found in all of /kaggle/input/: {len(images)}")
if images:
    print("Sample image path:", images[0])

print(f"\nTotal ZIP files found: {len(zips)}")
if zips:
    print("Sample zip path:", zips[0])

# Also, let's list all the datasets currently attached to your notebook
print("\nFolders in /kaggle/input/:")
for folder in os.listdir('/kaggle/input/'):
    print("-", folder)

In [ ]:
import os
import glob
import shutil
import random
from xml.etree import ElementTree as ET

# 1. Search the ENTIRE /kaggle/input/ directory for files
print("Searching entire /kaggle/input/ for XML and Image files...")

xml_files = glob.glob('/kaggle/input/**/*.xml', recursive=True)
img_files = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG'):
    img_files.extend(glob.glob(os.path.join('/kaggle/input/', '**/' + ext), recursive=True))

print(f"Found {len(xml_files)} XML files.")
print(f"Found {len(img_files)} Image files.")

output_dir = '/kaggle/working/yolo_dataset'

# Clear output directory if it exists
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Create YOLO folder structure
os.makedirs(f'{output_dir}/images/train', exist_ok=True)
os.makedirs(f'{output_dir}/images/val', exist_ok=True)
os.makedirs(f'{output_dir}/labels/train', exist_ok=True)
os.makedirs(f'{output_dir}/labels/val', exist_ok=True)

if len(xml_files) > 0 and len(img_files) > 0:
    # Create a dictionary to quickly find images by their base name
    img_dict = {os.path.basename(p).split('.')[0]: p for p in img_files}

    # 2. Define classes
    classes = {'helmet': 0, 'head': 1, 'Hardhat': 0, 'Head': 1} 

    # Shuffle for random 80/20 train/val split
    random.shuffle(xml_files)
    train_split = int(len(xml_files) * 0.8)

    def convert_box(size, box):
        dw = 1.0 / size[0]
        dh = 1.0 / size[1]
        x = (box[0] + box[2]) / 2.0
        y = (box[1] + box[3]) / 2.0
        w = box[2] - box[0]
        h = box[3] - box[1]
        return (x * dw, y * dh, w * dw, h * dh)

    # 3. Process each XML file
    train_count = 0
    val_count = 0

    for i, xml_file in enumerate(xml_files):
        split = 'train' if i < train_split else 'val'
        
        base_name = os.path.basename(xml_file).replace('.xml', '')
        
        if base_name not in img_dict:
            continue
            
        img_path = img_dict[base_name]
        
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find('size')
        
        if size is None:
            continue
            
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        
        label_lines = []
        for obj in root.iter('object'):
            class_name = obj.find('name').text.strip()
            if class_name not in classes:
                continue
            class_id = classes[class_name]
            
            xml_box = obj.find('bndbox')
            b = (
                float(xml_box.find('xmin').text), float(xml_box.find('ymin').text),
                float(xml_box.find('xmax').text), float(xml_box.find('ymax').text)
            )
            yolo_box = convert_box((w, h), b)
            label_lines.append(f"{class_id} {' '.join(['%.6f' % v for v in yolo_box])}")
        
        with open(f'{output_dir}/labels/{split}/{base_name}.txt', 'w') as f:
            f.write('\n'.join(label_lines))
            
        shutil.copy(img_path, f'{output_dir}/images/{split}/{os.path.basename(img_path)}')
        
        if split == 'train':
            train_count += 1
        else:
            val_count += 1

    print("\n✅ Conversion complete!")
    print(f"Train images: {train_count}")
    print(f"Val images: {val_count}")
else:
    print("❌ ERROR: Still couldn't find files.")

In [ ]:
import yaml

dataset_info = {
    'train': '/kaggle/working/yolo_dataset/images/train',
    'val': '/kaggle/working/yolo_dataset/images/val',
    'nc': 2,  # Number of classes
    'names': ['helmet', 'head'] # Standard classes for this dataset
}

with open('/kaggle/working/data.yaml', 'w') as outfile:
    yaml.dump(dataset_info, outfile, default_flow_style=False)

print("data.yaml created successfully!")

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8s model
model = YOLO('yolov8s.pt') 

# Train the model
results = model.train(
    data='/kaggle/working/data.yaml', 
    epochs=30, 
    imgsz=640, 
    batch=16, 
    name='helmet_detection_model'
)

print("🎉 Training Complete! Your model is saved in /kaggle/working/runs/detect/helmet_detection_model/weights/best.pt")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob
import os

# Load your newly trained model
model = YOLO('/kaggle/working/runs/detect/helmet_detection_model/weights/best.pt')

# Run prediction on an image
test_image = '/kaggle/input/notebooks/manarsaber/safety-helmet-detection-yolo8/hard_hat_yolo/test/images/hard_hat_workers1004.png' 

# Predict and save the result
results = model.predict(source=test_image, conf=0.4, save=True)

# Get the folder where YOLO saved the results
save_dir = results[0].save_dir

# Find ANY image file (.png, .jpg, or .jpeg) inside that folder
saved_images = glob.glob(os.path.join(save_dir, '*.png')) + glob.glob(os.path.join(save_dir, '*.jpg'))

if saved_images:
    # Display the first image found
    print(f"Found saved image: {saved_images[0]}")
    display(Image(filename=saved_images[0]))
else:
    print("Could not find the saved image.")

In [ ]:
from ultralytics import YOLO

# 1. Load your trained model
model = YOLO('/kaggle/working/runs/detect/helmet_detection_model/weights/best.pt')

# 2. PUT YOUR VIDEO PATH HERE!
# This is the path to the video you want to test.
video_path = '/kaggle/input/datasets/anishpatil2853/video2/gettyimages-1331230450-640_adpp.mp4' 

# 3. Run the prediction (YOLO will process the video and save the output)
results = model.predict(source=video_path, conf=0.4, save=True)
print("Prediction complete! Video saved.")

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os
import glob

# Automatically find the folder where YOLO just saved the video
predict_folders = glob.glob('/kaggle/working/runs/detect/predict*')
latest_folder = max(predict_folders, key=os.path.getctime)

# Find the video file inside that folder
video_files = glob.glob(os.path.join(latest_folder, '*.mp4')) + glob.glob(os.path.join(latest_folder, '*.avi'))

# Display the video
if video_files:
    processed_video_path = video_files[0]
    print(f"Found video: {processed_video_path}")
    
    with open(processed_video_path, 'rb') as f:
        video_encoded = b64encode(f.read()).decode('utf-8')
    
    video_html = f"""
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_encoded}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """
    display(HTML(video_html))
else:
    print("Could not find the output video.")